## Garch_M Model 


In [3]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import yfinance as yf

data = yf.download("NVDA", start="2006-01-01", end="2025-12-31")
data.to_csv("nvidia_stock.csv")

# Taking log of closing prices
log_returns = np.log(data['Close']).diff().dropna()

[*********************100%***********************]  1 of 1 completed


In [15]:
# Define tracking variables from log returns

# Defining log returns and number of observations 
data = log_returns.values
n_obs = len(data)

# Use squared returns to represent initial volatility
init_var = np.mean(data**2) 

# Define the GARCH-M(1,1) Negative Log-Likelihood function
def garch_m_loglike(parameters, data):
    """
    parameters: array containing [mu, lambda (risk premium), omega, alpha, beta]
    data: array of asset returns
    """
    mu, lam, omega, alpha, beta = parameters
    T = len(data)
    
    # Creating arrays for conditional variance and residuals
    sigma2 = np.zeros(T)
    epsilon = np.zeros(T)
    
    # squared returns initial variance
    sigma2[0] = init_var
    epsilon[0] = data[0] - (mu + lam * np.sqrt(sigma2[0]))
    
    log_likelihood = 0
    
    for t in range(1, T):
        # Volatility/Variance equation
        sigma2[t] = omega + alpha * (epsilon[t-1]**2) + beta * sigma2[t-1]
        
        # Mean equation (using conditional standard deviation)
        epsilon[t] = data[t] - (mu + lam * np.sqrt(sigma2[t]))
        
        # Normal distribution log-likelihood component for time t
        log_likelihood += -0.5 * np.log(2 * np.pi) - 0.5 * np.log(sigma2[t]) - 0.5 * (epsilon[t]**2) / sigma2[t]
        
    # Return negative log-likelihood for minimization
    return -log_likelihood

# Set starting parameters and constraints (Explicitly cast to float)
mu_init = float(np.mean(log_returns))
omega_init = float(init_var * 0.1)

initial_params = [mu_init, 0.1, omega_init, 0.1, 0.8] # 1D list of float values

# Parameter bounds to avoid division by zero or negative variance
bounds = [
    (None, None),  # mu
    (None, None),  # lambda
    (1e-6, None),  # omega
    (0.0, 1.0),    # alpha
    (0.0, 1.0)     # beta
]

# Fixed Constraint: alpha (index 3) + beta (index 4) < 1
def stationarity_constraint(parameters):
    return 0.999 - (parameters[3] + parameters[4])

constraints = {'type': 'ineq', 'fun': stationarity_constraint}

# Fit the GARCH-M Model
result = minimize(
    fun=garch_m_loglike, 
    x0=initial_params, 
    args=(data,),  
    bounds=bounds, 
    constraints=constraints,
    method='SLSQP'
) 


# Mapping Outputs

# Fitted Parameter Values
fitted_params = {
    'mu': result.x[0],
    'lambda (risk premium)': result.x[1],
    'omega': result.x[2],
    'alpha': result.x[3],
    'beta': result.x[4]
}

#  Model Selection Metrics

log_like = -result.fun  
k = len(initial_params)  

aic = 2 * k - 2 * log_like
bic = k * np.log(n_obs) - 2 * log_like

# 1-Day Volatility Forecast
final_mu, final_lam, final_omega, final_alpha, final_beta = result.x

sigma2_hist = np.zeros(n_obs)
epsilon_hist = np.zeros(n_obs)
sigma2_hist[0] = init_var
epsilon_hist[0] = data[0] - (final_mu + final_lam * np.sqrt(sigma2_hist[0]))

for t in range(1, n_obs):
    sigma2_hist[t] = final_omega + final_alpha * (epsilon_hist[t-1]**2) + final_beta * sigma2_hist[t-1]
    epsilon_hist[t] = data[t] - (final_mu + final_lam * np.sqrt(sigma2_hist[t]))

# Calculate future 1-day variance and take square root for volatility
forecast_variance_t1 = final_omega + final_alpha * (epsilon_hist[-1]**2) + final_beta * sigma2_hist[-1]
forecast_volatility_t1 = np.sqrt(forecast_variance_t1)

# Map to the next chronological date using log_returns index
future_date = log_returns.index[-1] + pd.Timedelta(days=1)
volatility_forecast = pd.Series([forecast_volatility_t1], index=[future_date])


# Display Results

print("\n--Fitted Parameters--")
for key, val in fitted_params.items():
    print(f"{key}: {val:.6f}")

print("\n--Model Ranking Metrics--")
print(f"Log-Likelihood: {log_like:.4f}")
print(f"AIC:            {aic:.4f}")
print(f"BIC:            {bic:.4f}")

print("\n--1-Day Forecasted Volatility--")
print(volatility_forecast)



--Fitted Parameters--
mu: 0.001286
lambda (risk premium): 0.100000
omega: 0.000096
alpha: 0.100000
beta: 0.800000

--Model Ranking Metrics--
Log-Likelihood: 10664.3966
AIC:            -21318.7931
BIC:            -21286.1783

--1-Day Forecasted Volatility--
2025-12-31    0.025087
dtype: float64


/var/folders/41/z1v1y3vd3hj0dn1s8t7dpssh0000gn/T/ipykernel_91033/838062169.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  epsilon[0] = data[0] - (mu + lam * np.sqrt(sigma2[0]))
/var/folders/41/z1v1y3vd3hj0dn1s8t7dpssh0000gn/T/ipykernel_91033/838062169.py:38: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  epsilon[t] = data[t] - (mu + lam * np.sqrt(sigma2[t]))
/var/folders/41/z1v1y3vd3hj0dn1s8t7dpssh0000gn/T/ipykernel_91033/838062169.py:103: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (